# Feature — KTO Training

In [ ]:
"""
REAL KTO — preference training from unpaired binary feedback, on the GPU.
========================================================================

DPO needs paired {chosen, rejected} for the same prompt. KTO (Kahneman-Tversky Optimization)
needs only a binary thumbs-up / thumbs-down on individual completions — closer to the signal a
deployed agent actually collects. This trains a real TRL `KTOTrainer` (LoRA) on SmolLM2-360M:
contract answers are labelled desirable, prose answers undesirable, unpaired.

Requires a GPU and SmolLM2-360M in the local Hugging Face cache.
"""
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

In [2]:
import sys
sys.modules["vllm"] = None  # env vllm is ABI-broken vs torch; TRL imports it eagerly

In [3]:
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import KTOTrainer, KTOConfig

In [4]:
MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"
OUT_DIR = "/tmp/agenttune-kto-student"
PROBE = "Classify the sentiment: 'terrible, broke on day one'"

In [5]:
def generate(model, tok, prompt):
    enc = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(model.device)
    with torch.no_grad():
        o = model.generate(**enc, max_new_tokens=14, do_sample=False,
                           pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(o[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [6]:
def main():
    print(f"[gpu]     {torch.cuda.get_device_name(0)}  (cuda: {torch.cuda.is_available()})")

    reviews = [
        ("I love this product, best purchase ever", "positive"),
        ("absolutely fantastic experience", "positive"),
        ("exceeded my expectations", "positive"),
        ("delighted with the quality", "positive"),
        ("this is the worst thing I've bought", "negative"),
        ("terrible, broke on day one", "negative"),
        ("support was rude and unhelpful", "negative"),
        ("waste of money, deeply disappointed", "negative"),
        ("it arrived on time, works as described", "neutral"),
        ("standard packaging, nothing special", "neutral"),
        ("does the job, no strong feelings", "neutral"),
        ("average product, meets the basics", "neutral"),
    ]
    # Unpaired binary feedback: contract answer = desirable, prose answer = undesirable.
    rows = []
    for text, lbl in reviews:
        prompt = [{"role": "user", "content": f"Classify the sentiment: '{text}'"}]
        rows.append({"prompt": prompt,
                     "completion": [{"role": "assistant", "content": f"SENTIMENT={lbl}"}],
                     "label": True})
        rows.append({"prompt": prompt,
                     "completion": [{"role": "assistant", "content": f"The sentiment of this review is {lbl}."}],
                     "label": False})
    print(f"[data]    {len(rows)} unpaired rows ({sum(r['label'] for r in rows)} desirable / "
          f"{sum(not r['label'] for r in rows)} undesirable)")

    tok = AutoTokenizer.from_pretrained(MODEL)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16).to("cuda")
    before = generate(base, tok, PROBE)
    del base
    torch.cuda.empty_cache()

    cfg = KTOConfig(
        output_dir=OUT_DIR, num_train_epochs=12, per_device_train_batch_size=4,
        learning_rate=1e-4, beta=0.1, max_length=128, desirable_weight=1.0, undesirable_weight=1.0,
        logging_steps=2, save_strategy="no", report_to=[], bf16=True,
    )
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16).to("cuda")
    trainer = KTOTrainer(
        model=model, ref_model=None, args=cfg, train_dataset=Dataset.from_list(rows),
        processing_class=tok,
        peft_config=LoraConfig(r=16, lora_alpha=32, task_type="CAUSAL_LM",
                               target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]),
    )
    trainer.train()
    trainer.save_model(OUT_DIR)

    def last(metric):
        vals = [h[metric] for h in trainer.state.log_history if metric in h]
        return vals[-1] if vals else float("nan")
    r_chosen, r_rejected = last("rewards/chosen"), last("rewards/rejected")

    del trainer, model
    torch.cuda.empty_cache()
    policy = PeftModel.from_pretrained(
        AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16).to("cuda"),
        OUT_DIR).merge_and_unload()
    after = generate(policy, tok, PROBE)

    print(f"[kto]     implicit reward — desirable {r_chosen:+.2f} vs undesirable {r_rejected:+.2f} "
          f"(margin {r_chosen - r_rejected:+.2f})")
    print(f"[probe]   {PROBE}")
    print(f"[before]  base policy -> {before!r}")
    print(f"[after]   KTO policy  -> {after!r}")
    print(f"[verdict] KTO scores desirable above undesirable: {r_chosen > r_rejected}; "
          f"greedy output moved off prose: {before.strip() != after.strip()}")

In [7]:
if __name__ == "__main__":
    main()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


[gpu]     NVIDIA L4  (cuda: True)
[data]    24 unpaired rows (12 desirable / 12 undesirable)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/tmp/ipykernel_382040/604548474.py:39: FutureWarning: The `KTOConfig` is now located in `trl.experimental`. Please update your imports to `from trl.experimental.kto import KTOConfig`. For more information, see https://github.com/huggingface/trl/issues/4223. Promoting KTO to the stable API is a high-priority task. Until then, this current path (`from trl import KTOConfig`) will remain, but API changes may occur.
  cfg = KTOConfig(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

Assembling KL train dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Step,Training Loss
2,0.499522
4,0.485212
6,0.467347
8,0.437886
10,0.401087
12,0.367623
14,0.331038
16,0.295761
18,0.264008
20,0.238635


peft/utils/save_and_load.py:438: UserWarning: Could not find a config file in HuggingFaceTB/SmolLM2-360M-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[kto]     implicit reward — desirable +1.94 vs undesirable -4.78 (margin +6.72)
[probe]   Classify the sentiment: 'terrible, broke on day one'
[before]  base policy -> 'The sentiment of the given text is negative.'
[after]   KTO policy  -> 'SENTIMENT=negative'
[verdict] KTO scores desirable above undesirable: True; greedy output moved off prose: True
